**This notebook is for demonstration purposes only. The focus is on illustrating the ideas rather than code implementation details.** For pseudocode and well-organized implementations, please refer to the textbook and the AIMA Python repository:
https://github.com/aimacode/aima-python/tree/master

In [ ]:
# ============================
# AC-3 visualization for the Mon/Tue/Wed CSP (A-G graph)
# - Implements AC-3 + Revise exactly (binary constraint: X != Y)
# - Step-by-step visualization with queue actions & domain deletions
# - Scrollable right log, newest entry on top
# ============================

import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
import matplotlib.patheffects as pe
import ipywidgets as widgets
from IPython.display import display, clear_output

# ----------------------------
# 1) Graph (same as your Mon/Tue/Wed picture)
# ----------------------------
VARS = list("ABCDEFG")
VALUES = ["Mon", "Tue", "Wed"]
VAL_SHORT = {"Mon":"M", "Tue":"T", "Wed":"W"}
VAL_COLOR = {"Mon":"#c84c4c", "Tue":"#3fa65a", "Wed":"#3f7fbf"}

edges = [
    ("A","B"), ("A","C"), ("B","C"),
    ("B","D"), ("D","E"), ("B","E"),
    ("C","E"), ("C","F"),
    ("E","F"), ("F","G"), ("E","G"),
]

G = nx.Graph()
G.add_nodes_from(VARS)
G.add_edges_from(edges)

# layout (spaced out)
pos = {
    "A": (3.0, 5.2),
    "B": (1.8, 4.0),
    "C": (4.2, 4.0),
    "D": (0.9, 2.6),
    "E": (2.6, 1.8),
    "F": (5.2, 2.7),
    "G": (6.2, 1.2),
}

# ----------------------------
# 2) CSP constraint: neighbors must be different (X != Y)
# ----------------------------
def constraint_neq(x, y):
    return x != y

# ----------------------------
# 3) AC-3 + Revise (records steps)
# ----------------------------
def all_arcs(graph):
    # directed arcs (X,Y) for every undirected edge {X,Y}
    arcs = []
    for u, v in graph.edges():
        arcs.append((u, v))
        arcs.append((v, u))
    return arcs

def revise(domains, X, Y):
    """
    Revise(domains, X, Y):
      revised = False
      for x in X.domain:
        if no y in Y.domain satisfies constraint(X,Y):
          delete x
          revised = True
      return revised, removed_values
    """
    revised = False
    removed = []

    # iterate over a copy because we may delete
    for x in list(domains[X]):
        # exists y support?
        supported = any(constraint_neq(x, y) for y in domains[Y])
        if not supported:
            domains[X].remove(x)
            revised = True
            removed.append(x)
    return revised, removed

def ac3_steps(initial_domains):
    """
    Returns: steps list
    Each step dict contains:
      - domains snapshot
      - queue snapshot (list of arcs)
      - current_arc (X,Y) or None
      - highlight_edges: set of undirected edges to highlight (current arc)
      - revised_info: (X, removed_values) or None
      - log html string
    """
    domains = {v: list(initial_domains[v]) for v in VARS}
    queue = all_arcs(G)

    steps = []

    def snap(log, current_arc=None, revised_info=None):
        hi = set()
        if current_arc is not None:
            x, y = current_arc
            hi.add(tuple(sorted((x, y))))
        steps.append({
            "domains": {k: list(v) for k, v in domains.items()},
            "queue": list(queue),
            "current_arc": current_arc,
            "highlight_edges": hi,
            "revised_info": revised_info,
            "log": log
        })

    snap("Initialize queue with all arcs.")

    while queue:
        (X, Y) = queue.pop(0)  # Dequeue
        snap(f"Dequeue arc (<b>{X}</b>, <b>{Y}</b>).", current_arc=(X, Y))

        revised, removed = revise(domains, X, Y)

        if revised:
            snap(
                f"Revise(<b>{X}</b>, <b>{Y}</b>) removed: <b>{', '.join(removed)}</b> from D({X}).",
                current_arc=(X, Y),
                revised_info=(X, removed)
            )

            if len(domains[X]) == 0:
                snap(f"❌ Domain of <b>{X}</b> becomes empty. AC-3 returns <b>false</b>.", current_arc=(X, Y))
                return steps, False

            # for each Z in X.neighbors - {Y}: Enqueue(Z, X)
            for Z in G.neighbors(X):
                if Z == Y:
                    continue
                queue.append((Z, X))
                snap(f"Enqueue arc (<b>{Z}</b>, <b>{X}</b>) because D({X}) changed.", current_arc=(Z, X))

        else:
            snap(f"Revise(<b>{X}</b>, <b>{Y}</b>) made no change.", current_arc=(X, Y))

    snap("Queue empty. AC-3 returns <b>true</b> (arc consistency enforced).")
    return steps, True

# ----------------------------
# 4) Two initial-domain modes (for teaching)
# ----------------------------
def domains_all_three():
    # This will produce NO deletions under X != Y (good to show the 'no effect' case)
    return {v: VALUES[:] for v in VARS}

def domains_with_demo_restrictions():
    # Add a few unary restrictions so AC-3 actually deletes values (for illustration).
    # You can change these to match your lecture story.
    D = {v: VALUES[:] for v in VARS}
    D["A"] = ["Mon"]          # A fixed to Mon
    D["G"] = ["Tue"]   # G cannot be Mon
    # (This combo usually triggers deletions along arcs into neighbors of A, etc.)
    return D

# choose initial domains here:
# INIT = domains_all_three()
INIT = domains_with_demo_restrictions()  # <-- change to domains_all_three() to show "no deletions"

STEPS, AC3_OK = ac3_steps(INIT)

# ----------------------------
# 5) Drawing helpers (dark, clean, domains as 3 slots with removed greyed)
# ----------------------------
BG = "#0b0b0b"
EDGE = "#e6e6e6"

def draw_domain_slots(ax, center, node_r, remaining_vals):
    """
    Always draw 3 slots (Mon/Tue/Wed).
    If removed, show greyed box.
    """
    cx, cy = center
    w = node_r * 1.05
    h = node_r * 0.36
    gap = node_r * 0.18

    y0 = cy - node_r - h - node_r*0.22
    total_w = 3*w + 2*gap
    x0 = cx - total_w/2

    for i, v in enumerate(VALUES):
        alive = (v in remaining_vals)
        face = VAL_COLOR[v] if alive else "#333333"
        alpha = 1.0 if alive else 0

        ax.add_patch(Rectangle(
            (x0 + i*(w+gap), y0), w, h,
            facecolor=face, alpha=alpha,
            edgecolor="#111111",
            linewidth=0.8, zorder=2
        ))
        ax.text(x0 + i*(w+gap) + w/2, y0 + h/2, VAL_SHORT[v],
                ha="center", va="center",
                fontsize=8.5, color="white", fontweight="bold",
                zorder=3,
                alpha=alpha,
                path_effects=[pe.withStroke(linewidth=1.6, foreground="black")])

def draw_step(step, ax):
    ax.clear()
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_facecolor(BG)

    domains = step["domains"]
    hi_edges = {tuple(sorted(e)) for e in step["highlight_edges"]}
    current_arc = step["current_arc"]
    revised_info = step["revised_info"]

    node_r = 0.3

    # edges (thin)
    for (u, v) in G.edges():
        e = tuple(sorted((u, v)))
        x1, y1 = pos[u]
        x2, y2 = pos[v]

        if e in hi_edges:
            ax.plot([x1, x2], [y1, y2],
                    linewidth=3.0, color="#f0a500",
                    solid_capstyle="round", zorder=0)
        else:
            ax.plot([x1, x2], [y1, y2],
                    linewidth=1.6, color=EDGE,
                    solid_capstyle="round", zorder=0)

    # nodes
    revised_node = revised_info[0] if revised_info else None

    for n in VARS:
        x, y = pos[n]

        # node fill: if singleton, fill with that value color; else white
        fill = "white"
        if len(domains[n]) == 1:
            fill = VAL_COLOR[domains[n][0]]

        # border: revised node red border in that step
        border = "#d62728" if (revised_node == n) else "#0aa3d1"
        lw = 2.0 if (revised_node == n) else 1.6

        circ = Circle((x, y), radius=node_r, facecolor=fill,
                      edgecolor=border, linewidth=lw, zorder=4)
        ax.add_patch(circ)

        # label
        txt = ax.text(x, y, n, ha="center", va="center",
                      fontsize=14, fontweight="bold",
                      color="#111111", zorder=5)
        txt.set_path_effects([pe.withStroke(linewidth=3.0, foreground="white")])

        # domains (3 fixed slots)
        draw_domain_slots(ax, (x, y), node_r=node_r, remaining_vals=domains[n])

    # annotate current arc direction
    if current_arc is not None:
        X, Y = current_arc
        ax.text(0.02, 0.98, f"Current arc: ({X}, {Y})",
                transform=ax.transAxes, ha="left", va="top",
                fontsize=11, color="#f0a500",
                path_effects=[pe.withStroke(linewidth=3, foreground="black")])

    xs = [pos[n][0] for n in VARS]
    ys = [pos[n][1] for n in VARS]

        # ---- draw current queue on the LEFT plot ----
    queue = step["queue"]
    max_show = 99  # show only first k arcs

    if queue:
        lines = ["Queue:"]
        for (u, v) in queue[:max_show]:
            lines.append(f"({u},{v})")
        if len(queue) > max_show:
            lines.append(f"... (+{len(queue)-max_show})")

        q_text = "\n".join(lines)

        ax.text(
            0.02, 0.85, q_text,
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=11,
            family="monospace",
            color="#eaeaea",
            bbox=dict(
                boxstyle="round,pad=0.35",
                facecolor="#1a1a1a",
                edgecolor="#666666",
                linewidth=1.0
            ),
            zorder=10,
            path_effects=[pe.withStroke(linewidth=3, foreground="black")]
        )

    ax.set_xlim(min(xs)-2, max(xs)+1.2)
    ax.set_ylim(min(ys)-1.4, max(ys)+1.0)

# ----------------------------
# 6) Scrollable RIGHT log (newest on top) + show queue head a bit
# ----------------------------
def log_panel_html(step_idx, steps, window=250, queue_preview=10):
    start = max(0, step_idx - window + 1)

    # newest first
    items = []
    for i in range(step_idx, start - 1, -1):
        q = steps[i]["queue"]
        qtxt = ", ".join([f"({a},{b})" for (a,b) in q[:queue_preview]])
        more = "" if len(q) <= queue_preview else f" … (+{len(q)-queue_preview})"
        items.append(
            f"<div style='margin:6px 0;'>"
            f"<div><span style='color:#888;'><b>{i}.</b></span> {steps[i]['log']}</div>"
            f"<div style='color:#aaa; font-size:11px; margin-top:2px;'>"
            f"queue head: [{qtxt}]{more}"
            f"</div>"
            f"</div>"
        )

    return f"""
    <div style="
        font-family: -apple-system, Segoe UI, Arial;
        font-size: 12px;
        line-height: 1.25;
        height: 560px;
        overflow-y: auto;
        border: 1px solid #333;
        border-radius: 12px;
        padding: 10px;
        background: #0f0f0f;
        color: #eaeaea;
    ">
      {''.join(items)}
    </div>
    """

# ----------------------------
# 7) UI
# ----------------------------
out_left = widgets.Output()
out_right = widgets.HTML()

slider = widgets.IntSlider(
    value=0, min=0, max=len(STEPS)-1, step=1,
    description="Step", continuous_update=False,
    layout=widgets.Layout(width="520px")
)
btn_prev = widgets.Button(description="Prev", layout=widgets.Layout(width="80px"))
btn_next = widgets.Button(description="Next", layout=widgets.Layout(width="80px"))
play = widgets.Play(interval=650, value=0, min=0, max=len(STEPS)-1, step=1)
widgets.jslink((play, "value"), (slider, "value"))

def render(i):
    with out_left:
        clear_output(wait=True)
        fig = plt.figure(figsize=(10.2, 6.2))
        ax = fig.add_subplot(1,1,1)
        draw_step(STEPS[i], ax)
        plt.show()

    out_right.value = log_panel_html(i, STEPS, window=300, queue_preview=10)

def on_slider(change):
    render(change["new"])

def on_prev(_):
    slider.value = max(slider.min, slider.value - 1)

def on_next(_):
    slider.value = min(slider.max, slider.value + 1)

slider.observe(on_slider, names="value")
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

controls = widgets.HBox([play, slider, btn_prev, btn_next])
layout = widgets.HBox(
    [out_left, widgets.Box([out_right], layout=widgets.Layout(width="460px"))],
    layout=widgets.Layout(align_items="flex-start", justify_content="space-between")
)

display(controls, layout)
render(0)

print("AC-3 returned:", AC3_OK)
# print("Tip: set INIT = domains_all_three() to show the 'no deletions' case.")


AC-3 returned: True
